# CEG-WM development exploration

This output-free Notebook is the only development entrypoint currently authorized for **Run all**. It is a thin Colab and Google Drive entrypoint delivered by a later delivery revision, while every executable project file is fetched from the independently approved immutable `EXECUTION_REVISION` below. A mutable branch must never replace that execution authority. The repository server owns environment setup, model retrieval, formal runner execution, records, recovery, and result or diagnostic packaging. Drive export copies are delivery conveniences; scientific completion is determined only by verified `COMMITTED` bundles under the persistent root. `experiment_execution.ipynb` and `runtime_qualification.ipynb` are paused and are not authorized to run.

The frozen mechanism-screening budget is 240 scientific units plus 42 operational units, 282 total, with 846 maximum attempts. This first Run all is restricted to two preflight clusters plus two wiring clusters: 4 operational units and 0 scientific units. It does not count toward module science. A complete screening session may be enabled only by a later independently reviewed entrypoint after Agent2 and Agent3 verify this preflight.

The prior `ceg_wm_development_exploration_detector_crossfit_execution` run and all of its scientific records, operational records, and diagnostic artifacts remain immutable. The prior `ceg_wm_development_exploration_scientific_execution` run also remains immutable: two operational commits, zero scientific commits, dangling unit 0002 attempt 0, and diagnostic `builtins.AssertionError`. The prior `ceg_wm_development_exploration_science_first_v42` run namespace and any `ceg_wm_development_exploration_joint_record_execution` directory also remain untouched. Existing records, dangling attempts, and full artifacts are never read, migrated, rewritten, or deleted by this new run.


In [ ]:
from google.colab import drive, userdata
from datetime import datetime, timezone
from hashlib import sha256
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys

drive.mount('/content/drive')


In [ ]:
REPOSITORY_URL = 'https://github.com/RICHAAARC/CEG-WM.git'
EXECUTION_REVISION = '2ff836f45c4012010092f7075e749507ae2ad9ae'
RUN_ID = 'ceg_wm_thirteen_module_mechanism_screening'
SESSION_ID = datetime.now(timezone.utc).strftime('colab_%Y%m%dt%H%M%S%fz')
CHECKOUT_ROOT = Path(f'/content/ceg_wm_development_checkout_{SESSION_ID}')
CACHE_ROOT = Path('/content/ceg_wm_development_cache')
DRIVE_MOUNT = Path('/content/drive').resolve()
DRIVE_ROOT = DRIVE_MOUNT / 'MyDrive' / 'CEG-WM' / 'development_exploration'
PERSISTENT_ROOT = DRIVE_ROOT / 'persistent'
EXPORT_ROOT = DRIVE_ROOT / 'exports' / EXECUTION_REVISION / RUN_ID / SESSION_ID
PERSISTENT_ROOT.mkdir(parents=True, exist_ok=True)
assert DRIVE_MOUNT in PERSISTENT_ROOT.resolve().parents
probe_path = PERSISTENT_ROOT / f'.write_probe_{SESSION_ID}'
with probe_path.open('x', encoding='utf-8') as probe:
    probe.write('development exploration persistent root available\n')
probe_path.unlink()
secret_environment = os.environ.copy()
secret_environment['HF_TOKEN'] = userdata.get('HF_TOKEN')
secret_environment['CEG_WM_ROOT_KEY'] = userdata.get('CEG_WM_ROOT_KEY')
assert secret_environment['HF_TOKEN'] and secret_environment['CEG_WM_ROOT_KEY']


In [ ]:
CHECKOUT_ROOT.mkdir(parents=True, exist_ok=False)
subprocess.run(['git', '-C', str(CHECKOUT_ROOT), 'init'], check=True)
subprocess.run(['git', '-C', str(CHECKOUT_ROOT), 'remote', 'add', 'origin', REPOSITORY_URL], check=True)
subprocess.run(['git', '-C', str(CHECKOUT_ROOT), 'fetch', '--depth', '1', 'origin', EXECUTION_REVISION], check=True)
subprocess.run(['git', '-C', str(CHECKOUT_ROOT), 'checkout', '--detach', 'FETCH_HEAD'], check=True)
observed_revision = subprocess.run(['git', '-C', str(CHECKOUT_ROOT), 'rev-parse', 'HEAD'], check=True, capture_output=True, text=True).stdout.strip()
observed_status = subprocess.run(['git', '-C', str(CHECKOUT_ROOT), 'status', '--porcelain'], check=True, capture_output=True, text=True).stdout
assert observed_revision == EXECUTION_REVISION
assert observed_status == ''


In [ ]:
server_entrypoint = CHECKOUT_ROOT / 'scripts/experiment_execution/development_exploration_server.py'
command = [
    sys.executable, str(server_entrypoint),
    '--repository-root', str(CHECKOUT_ROOT),
    '--expected-revision', EXECUTION_REVISION,
    '--persistent-root', str(PERSISTENT_ROOT),
    '--cache-root', str(CACHE_ROOT),
    '--run-id', RUN_ID,
    '--session-id', SESSION_ID,
    '--maximum-wiring-clusters', '2',
]
process = subprocess.Popen(
    command,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    env=secret_environment,
)
assert process.stdout is not None
for log_line in process.stdout:
    print(log_line, end='')
server_exit_code = process.wait()
del secret_environment
success_receipt = PERSISTENT_ROOT / RUN_ID / 'server_receipts' / SESSION_ID / 'execution_receipt.json'
failure_receipt_root = PERSISTENT_ROOT / RUN_ID / 'server_failures' / SESSION_ID
receipt_candidates = ([success_receipt] if success_receipt.is_file() else []) + sorted(failure_receipt_root.glob('execution_failure_receipt_*.json'))
assert len(receipt_candidates) == 1, 'server did not persist one unique session receipt'
receipt_source = receipt_candidates[0]
receipt = json.loads(receipt_source.read_text(encoding='utf-8'))
assert receipt['committed_revision'] == EXECUTION_REVISION
assert receipt['run_id'] == RUN_ID and receipt['session_id'] == SESSION_ID
assert receipt['exit_code'] == server_exit_code


In [ ]:
def file_sha256(path):
    digest = sha256()
    with Path(path).open('rb') as source:
        for block in iter(lambda: source.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

def copy_to_drive_export(source, destination, expected_sha256):
    source = Path(source)
    destination = Path(destination)
    if destination.exists():
        raise RuntimeError('Drive export destination already exists')
    shutil.copyfile(source, destination)
    if file_sha256(destination) != expected_sha256:
        raise RuntimeError('Drive export SHA-256 mismatch')
    return destination

artifact_path = Path(receipt['artifact_path'])
assert not artifact_path.is_symlink()
artifact_source = artifact_path.resolve()
assert artifact_source.is_file()
assert PERSISTENT_ROOT.resolve() in artifact_source.parents
assert receipt['artifact_kind'] in {'development_exploration_result', 'development_exploration_diagnostic'}
assert receipt['scientific_claims_supported'] is False
assert receipt['calibration_locked'] is False
assert file_sha256(artifact_source) == receipt['artifact_sha256']
receipt_sha256 = file_sha256(receipt_source)
EXPORT_ROOT.mkdir(parents=True, exist_ok=False)
artifact_export = copy_to_drive_export(artifact_source, EXPORT_ROOT / artifact_source.name, receipt['artifact_sha256'])
receipt_export = copy_to_drive_export(receipt_source, EXPORT_ROOT / 'execution_receipt.json', receipt_sha256)
checksums_path = EXPORT_ROOT / 'SHA256SUMS'
with checksums_path.open('x', encoding='utf-8') as checksums:
    checksums.write(f"{receipt['artifact_sha256']}  {artifact_export.name}\n")
    checksums.write(f'{receipt_sha256}  {receipt_export.name}\n')
summary = {
    'artifact_kind': receipt['artifact_kind'],
    'artifact_path': str(artifact_export),
    'artifact_sha256': receipt['artifact_sha256'],
    'receipt_path': str(receipt_export),
    'receipt_sha256': receipt_sha256,
    'checksums_path': str(checksums_path),
    'committed_revision': EXECUTION_REVISION,
    'run_id': RUN_ID,
    'session_id': SESSION_ID,
    'committed_unit_count': receipt.get('committed_unit_count', 0),
    'termination_reason': receipt.get('termination_reason', receipt.get('failure_stage')),
}
print(json.dumps(summary, indent=2, sort_keys=True))
if server_exit_code != 0:
    raise RuntimeError('development exploration session ended with a diagnostic; Drive export is preserved')
